# Embedding and Positional Encoding

Models cannot understand words as we do. They need to be converted to tokens first then to numerical representations (embeddings) which the model can understand. Embedding is the first operation in the Transformer. In the 'Attention is all you need' paper, the embeddings are scaled by the square root of the model dimension (d_model). This is done to prevent the positional information to overpower the token information in the positional encodings

In [1]:
import torch
import math
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model) #the scaling

## Role of positional encoding

The transformer has limited capacity to keep track of word order as it processes words in parallel. Positional encodings allow the model to understand the position of tokens in a sequence when processing ensuring differences between A B C and C B A .

How to do this: Adding the word vector with the position vector
The positional encodings have the same dimensions as the embeddings so they can be summed

In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model) # create a matrix of zeros with the for all embeddings
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) # create a 1D tensor from 0 to max_sequence_length
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term) # selects all rows anad the even columns
        pe[:, 1::2] = torch.cos(position * div_term) # selects all rows and the odd columns
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)], self.pe[:, :x.size(1)]


In [16]:
vocab_size = 10
d_model = 4
tokens = torch.tensor([1,2,3])
tokens

tensor([1, 2, 3])

In [17]:
embedding_layer = InputEmbeddings(vocab_size=vocab_size, d_model=d_model)
embeddings =  embedding_layer(tokens)

In [18]:
embeddings.detach().numpy()

array([[-1.1544286 ,  1.0550823 , -2.27606   ,  0.6621352 ],
       [-0.0396321 ,  4.113746  , -1.558693  ,  0.59662884],
       [ 1.3692206 , -1.694793  , -1.6742321 ,  0.62369347]],
      dtype=float32)

In [19]:
pe_layer = PositionalEncoding(d_model=d_model, max_seq_length=3)
pe, enc = pe_layer(embeddings)


In [20]:
print('embeddings:', embeddings)
print('positional encodings:', enc)
print('summation:', pe )

embeddings: tensor([[-1.1544,  1.0551, -2.2761,  0.6621],
        [-0.0396,  4.1137, -1.5587,  0.5966],
        [ 1.3692, -1.6948, -1.6742,  0.6237]], grad_fn=<MulBackward0>)
positional encodings: tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])
summation: tensor([[[-1.1544,  2.0551, -2.2761,  1.6621],
         [ 0.8018,  4.6540, -1.5487,  1.5966],
         [ 2.2785, -2.1109, -1.6542,  1.6235]]], grad_fn=<AddBackward0>)


In [21]:
pe = pe.detach().numpy()

In [22]:
from ipywidgets import interact
import ipywidgets as widgets

In [28]:
def positional_encoding(d_model, max_seq_length):
    pe = torch.zeros(max_seq_length, d_model) # create a matrix of zeros with the for all embeddings
    position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) # create a 1D tensor from 0 to max_sequence_length
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term) # selects all rows anad the even columns
    pe[:, 1::2] = torch.cos(position * div_term) # selects all rows and the odd columns
    # self.register_buffer('pe', pe.unsqueeze(0))
    return pe

In [48]:
@interact(d_model=widgets.IntSlider(min=8, max=40, step=8, value=32), 
max_seq_length=widgets.IntSlider(min=8, max=50, step=4, value=50))
def plot_pe(d_model, max_seq_length):
    plt.close('all')
    pe = positional_encoding(d_model, max_seq_length)
    plt.imshow(pe.T, aspect='auto', cmap='RdBu');
    plt.colorbar()
    plt.xlabel('position')
    plt.ylabel('dimension')
    plt.show();

interactive(children=(IntSlider(value=32, description='d_model', max=40, min=8, step=8), IntSlider(value=50, d…